# Platrixa P5a — Specialist Finance Model Training

## Qwen2.5-1.5B-Instruct + LoRA on real accounting data

**Model:** `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit`
**Training data:** `specialist_clean_training.jsonl` (46 records)
**Max steps:** 20 (conservative — see comments below)

---

### What this notebook does
1. Installs Unsloth + dependencies
2. Loads the 1.5B Instruct model (NOT the 7B — T4 VRAM constraint)
3. Configures LoRA adapters
4. Loads and validates the real Platrixa accounting JSONL
5. Formats records into Alpaca prompt format
6. Trains with conservative settings
7. Tests inference on accounting prompts
8. Saves LoRA adapter locally + to Google Drive

### Critical configuration choices
- `load_in_4bit = False` — intentional. The 1.5B model fits in FP16 on T4 without 4-bit quantization.
- `max_steps = 20` — intentional. With only 46 training records, more steps cause overfitting.
- `max_seq_length = 1024` — accounting records are ~200-400 tokens; 1024 is sufficient.

## 1. Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. Package Sanity Check

In [ ]:
# --- Defensive: verify Unsloth imported successfully ---
try:
    import unsloth
    print(f"\u2705 Unsloth {unsloth.__version__} imported OK")
except ImportError as e:
    raise RuntimeError(
        f"\u274c Unsloth import failed: {e}\n"
        "Fix: Runtime \u2192 Restart runtime, then Run all again."
    )

import torch
print(f"\u2705 PyTorch {torch.__version__}")
print(f"\u2705 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("\u274c No GPU detected. Go to Runtime \u2192 Change runtime type \u2192 T4 GPU.")

# Check key packages
import transformers
import peft
import trl
print(f"\u2705 transformers {transformers.__version__}")
print(f"\u2705 peft {peft.__version__}")
print(f"\u2705 trl {trl.__version__}")

## 3. Model Loading

**Base model:** `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit`

We use the 1.5B Instruct model, NOT the 7B. Reasons:
- T4 GPU has ~15 GB VRAM — 7B + LoRA + training activations is fragile
- 1.5B fits comfortably with ~8 GB headroom on T4
- For the narrow task of "student text \u2192 structured interpretation JSON", 3B/7B is overkill
- The 46-record training set is the real bottleneck, not model size

**`load_in_4bit = False`** — The 1.5B model fits in FP16. Disabling 4-bit avoids bitsandbytes compatibility issues on some Colab T4 runtimes.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024  # Accounting records are ~200-400 tokens; 1024 is sufficient
dtype = None            # Auto-detect: FP16 on T4
load_in_4bit = False    # Intentional: 1.5B fits in FP16 on T4. Avoids bitsandbytes issues.

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

print(f"Loading model: {MODEL_NAME}")
print(f"  max_seq_length: {max_seq_length}")
print(f"  load_in_4bit: {load_in_4bit}")
print(f"  dtype: {dtype} (auto-detect)")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# --- Defensive: verify correct model loaded ---
actual_name = getattr(model.config, 'name_or_path', 'unknown')
print(f"\n\u2705 Model loaded: {actual_name}")
if "1.5B" not in actual_name and "1.5b" not in actual_name.lower():
    print(f"\u26a0\ufe0f  WARNING: Expected Qwen2.5-1.5B, got {actual_name}")
    print("   If this is intentional, ignore this warning.")

# Report VRAM usage
if torch.cuda.is_available():
    mem_allocated = torch.cuda.memory_allocated(0) / 1e9
    mem_reserved = torch.cuda.memory_reserved(0) / 1e9
    print(f"   VRAM allocated: {mem_allocated:.2f} GB")
    print(f"   VRAM reserved:  {mem_reserved:.2f} GB")

## 4. LoRA Configuration

Standard LoRA config for Qwen2.5 with Unsloth. r=16 is a good default for this model size.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,       # 0 is optimized for Unsloth
    bias = "none",          # "none" is optimized for Unsloth
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("\u2705 LoRA adapters configured")
model.print_trainable_parameters()

## 5. Data Prep — Real Platrixa Accounting Data

Load `specialist_clean_training.jsonl` from Colab's `/content/` directory.

**After runtime reset, re-upload this file** to Colab's Files panel.

In [ ]:
import os
import re
import json
from datasets import load_dataset

# --- 1. Verify file exists before loading ---
TRAINING_JSONL = "/content/specialist_clean_training.jsonl"

if not os.path.exists(TRAINING_JSONL):
    raise FileNotFoundError(
        f"\u274c Training file not found: {TRAINING_JSONL}\n"
        f"   Upload specialist_clean_training.jsonl to Colab's Files panel.\n"
        f"   After runtime resets, files in /content/ are deleted."
    )

# --- 2. Normalize filename (defensive) ---
fname = os.path.basename(TRAINING_JSONL)
if not re.match(r'^[a-zA-Z0-9_.-]+$', fname):
    print(f"\u26a0\ufe0f  Filename '{fname}' contains unusual characters. Renaming.")
    safe_fname = re.sub(r'[^a-zA-Z0-9_.-]', '_', fname)
    safe_path = os.path.join(os.path.dirname(TRAINING_JSONL), safe_fname)
    os.rename(TRAINING_JSONL, safe_path)
    TRAINING_JSONL = safe_path
    print(f"   Renamed to: {TRAINING_JSONL}")

# --- 3. Load dataset ---
dataset = load_dataset("json", data_files=TRAINING_JSONL, split="train")

# --- 4. Validate ---
print(f"File:          {TRAINING_JSONL}")
print(f"Record count:  {len(dataset)}")
print(f"Columns:       {dataset.column_names}")
print(f"Output type:   {type(dataset[0]['output']).__name__}")

# Check for empty fields (FIXED: previous version used r.get("","") which always returned 0)
empty_inst = sum(1 for r in dataset if not str(r.get("instruction", "")).strip())
empty_in   = sum(1 for r in dataset if not str(r.get("input", "")).strip())
empty_out  = sum(1 for r in dataset if not str(r.get("output", "")).strip())

print(f"Empty instruction: {empty_inst}/{len(dataset)}")
print(f"Empty input:       {empty_in}/{len(dataset)}")
print(f"Empty output:      {empty_out}/{len(dataset)}")

# Check output is valid JSON
bad_json = 0
for r in dataset:
    try:
        json.loads(r["output"])
    except (json.JSONDecodeError, TypeError):
        bad_json += 1

print(f"Invalid JSON outputs: {bad_json}/{len(dataset)}")

# --- 5. Show sample (not full dataset) ---
print(f"\n=== Sample record ===")
sample = dataset[0]
print(f"Instruction: {sample['instruction'][:120]}...")
print(f"Input:       {sample['input']}")
out = json.loads(sample['output'])
print(f"Output keys: {list(out.keys())}")

# --- 6. Confirm no Fibonacci/Alpaca data ---
fib_check = any("fibonacci" in str(r.get("input", "")).lower() for r in dataset)
alpaca_check = any("alpaca" in str(r.get("instruction", "")).lower() for r in dataset)
print(f"\nFibonacci records found: {fib_check}")
print(f"Alpaca records found:    {alpaca_check}")
if fib_check or alpaca_check:
    raise ValueError("\u274c Dataset contains non-accounting records. Check the JSONL file.")
print("\u2705 Dataset validated \u2014 real Platrixa accounting data only")

## 6. Formatting Function — Accounting Prompt

Converts each `{instruction, input, output}` record into the Alpaca prompt format.
The output JSON is preserved exactly as the training target.

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    """Convert Platrixa accounting records into Alpaca prompt format.

    Input record: {instruction, input, output}
    - instruction: task description for the model
    - input: raw student accounting text
    - output: JSON string with structured interpretation

    The output JSON is preserved exactly as the training target.
    Do NOT re-parse or modify the output JSON.
    """
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"\u2705 Formatted {len(dataset)} records")
print(f"   Columns after formatting: {dataset.column_names}")

# Show one formatted example (truncated)
sample_text = dataset[0]["text"]
print(f"\n=== Sample formatted prompt (first 500 chars) ===")
print(sample_text[:500])
print("...")

## 7. Train the Model

### Training configuration (intentional choices)

| Parameter | Value | Reason |
|-----------|-------|--------|
| `max_steps` | 20 | Conservative. 46 records x 1 epoch ~ 12 steps. 20 steps ~ 1.6 epochs. More causes overfitting. |
| `per_device_train_batch_size` | 2 | Fits in T4 VRAM with 1.5B model |
| `gradient_accumulation_steps` | 4 | Effective batch = 8 |
| `learning_rate` | 2e-4 | Standard for LoRA fine-tuning |
| `warmup_steps` | 5 | Brief warmup |

**Why not more steps?** With 46 training records, the model memorizes quickly. The 60-step config from earlier experiments showed overfitting. 20 steps is a conservative starting point — tune using held-out evaluation, not intuition.

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1,   # Uncomment for full epoch training
        max_steps = 20,          # Conservative: 46 records, ~1.6 epochs
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
print("\u2705 SFTTrainer configured")
print(f"   Dataset size: {len(dataset)} records")
print(f"   Max steps: {trainer.args.max_steps}")
print(f"   Batch size: {trainer.args.per_device_train_batch_size}")
print(f"   Grad accum: {trainer.args.gradient_accumulation_steps}")
print(f"   Effective batch: {trainer.args.per_device_train_batch_size * trainer.args.gradient_accumulation_steps}")

In [ ]:
trainer_stats = trainer.train()
print(f"\n\u2705 Training complete")
print(f"   Total steps: {trainer_stats.global_step}")
print(f"   Training loss: {trainer_stats.training_loss:.4f}")

## 8. Inference — Test the Trained Model

Run three accounting prompts to verify the model produces structured JSON output.

In [ ]:
# --- Test 1: Simple purchase ---
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Parse the student's accounting language into a grounded structured interpretation. Do not invent missing information.",
        "Purchased goods from Raj for Rs.20000 by cheque.",
        "",
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
result1 = tokenizer.batch_decode(outputs)
print("=== Test 1: Simple purchase ===")
print(result1[0][-300:])  # Print last 300 chars (the generated part)

In [ ]:
# --- Test 2: Expense ---
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Parse the student's accounting language into a grounded structured interpretation. Do not invent missing information.",
        "Paid electricity bill Rs.2800.",
        "",
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
result2 = tokenizer.batch_decode(outputs)
print("=== Test 2: Expense ===")
print(result2[0][-300:])

In [ ]:
# --- Test 3: Ambiguous case (cash vs credit) ---
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Parse the student's accounting language into a grounded structured interpretation. Do not invent missing information.",
        "Purchased goods from Raj for Rs.5000.",
        "",
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
result3 = tokenizer.batch_decode(outputs)
print("=== Test 3: Ambiguous (no payment method) ===")
print(result3[0][-300:])

## 9. Save the Model

Save LoRA adapter locally first, then persist to Google Drive.

In [ ]:
import os

# --- Save locally ---
LOCAL_SAVE_DIR = "qwen_platrixa_lora"
model.save_pretrained(LOCAL_SAVE_DIR)
tokenizer.save_pretrained(LOCAL_SAVE_DIR)

# Verify saved artifact exists
adapter_config = os.path.join(LOCAL_SAVE_DIR, "adapter_config.json")
if os.path.exists(adapter_config):
    print(f"\u2705 LoRA adapter saved locally: {LOCAL_SAVE_DIR}/")
    for f in os.listdir(LOCAL_SAVE_DIR):
        size = os.path.getsize(os.path.join(LOCAL_SAVE_DIR, f))
        print(f"   {f}: {size/1024:.1f} KB")
else:
    raise RuntimeError(f"\u274c Save failed \u2014 {adapter_config} not found")

## 10. Persist to Google Drive

Colab's local storage is ephemeral. After runtime disconnect, `/content/` is deleted.
This cell copies the LoRA adapter to Google Drive for durability.

In [ ]:
import shutil

# --- Mount Google Drive ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_SAVE_DIR = "/content/drive/MyDrive/platrixa_p5a_lora"

    # Copy local -> Drive
    if os.path.exists(DRIVE_SAVE_DIR):
        shutil.rmtree(DRIVE_SAVE_DIR)
    shutil.copytree(LOCAL_SAVE_DIR, DRIVE_SAVE_DIR)

    # Verify
    if os.path.exists(os.path.join(DRIVE_SAVE_DIR, "adapter_config.json")):
        print(f"\u2705 LoRA adapter persisted to Google Drive:")
        print(f"   {DRIVE_SAVE_DIR}/")
        for f in os.listdir(DRIVE_SAVE_DIR):
            size = os.path.getsize(os.path.join(DRIVE_SAVE_DIR, f))
            print(f"   {f}: {size/1024:.1f} KB")
    else:
        raise RuntimeError(f"\u274c Drive save failed \u2014 adapter_config.json not found in {DRIVE_SAVE_DIR}")

except ImportError:
    print("\u26a0\ufe0f  Google Colab not detected \u2014 skipping Drive persistence")
    print(f"   LoRA adapter is at: {os.path.abspath(LOCAL_SAVE_DIR)}/")
except Exception as e:
    print(f"\u274c Drive persistence failed: {e}")
    print(f"   LoRA adapter is still at: {os.path.abspath(LOCAL_SAVE_DIR)}/")

## 11. Load Saved LoRA for Later Inference

After restarting runtime, load the saved LoRA adapter instead of retraining.

In [ ]:
# Change to True to load a previously saved adapter
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_platrixa_lora",
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model)

    # Test with a saved adapter
    inputs = tokenizer(
    [
        alpaca_prompt.format(
            "Parse the student's accounting language into a grounded structured interpretation. Do not invent missing information.",
            "Purchased goods from Raj for Rs.20000 by cheque.",
            "",
        )
    ], return_tensors = "pt").to("cuda")

    from transformers import TextStreamer
    text_streamer = TextStreamer(tokenizer)
    _ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 256)

## 12. 4-Bit Quantization Investigation

### Current status
- `load_in_4bit = False` — intentional and working
- The 1.5B model fits in FP16 on T4 with ~8 GB headroom

### Why 4-bit was disabled
- bitsandbytes 4-bit quantizer sometimes dispatches modules to CPU/disk on Colab T4
- This happens when VRAM is fragmented or stale GPU state exists
- Restarting runtime often fixes it, but is fragile

### When to re-enable 4-bit
- If switching to a larger model (3B or 7B) where FP16 doesn't fit
- Requires: `bitsandbytes>=0.43.0`, `torch>=2.10`, CUDA 12.x
- Test with `load_in_4bit = True` + `max_seq_length = 512` first

### Tested compatible versions (as of Aug 2026)
- transformers 4.56.2
- trl 0.22.2
- peft (latest)
- unsloth 2026.8.22
- torch 2.11.0+cu128

## 13. Save to GGUF / llama.cpp (optional)

For local deployment or Ollama integration.

In [ ]:
# Uncomment to save as GGUF
# model.save_pretrained_gguf("qwen_platrixa", tokenizer,)
# model.save_pretrained_gguf("qwen_platrixa", tokenizer, quantization_method = "q4_k_m")
# model.save_pretrained_gguf("qwen_platrixa", tokenizer, quantization_method = "f16")

---

## Summary

| Item | Value |
|------|-------|
| Base model | `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit` |
| LoRA rank | 16 |
| Training data | `specialist_clean_training.jsonl` (46 records) |
| Max steps | 20 |
| load_in_4bit | False |
| max_seq_length | 1024 |
| Local save | `qwen_platrixa_lora/` |
| Drive save | `/content/drive/MyDrive/platrixa_p5a_lora/` |

### Next steps
1. Run this notebook in Colab on a T4 GPU
2. Check inference test outputs for structured JSON quality
3. Run the evaluation harness: `python scripts/fte_fyjc_p5a_evaluation.py`
4. Compare fine-tuned vs base model on held-out evaluation tiers
5. Tune `max_steps` based on evaluation performance, not intuition